In [ ]:
import time
import torch
from image_embeddings import build_models, load_image, download_image, parse_json_gz

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Device: cuda


In [ ]:
import os

META_PATH = 'meta_Toys_and_Games.json.gz'
IMG_DIR = 'data/images'
N = 10

os.makedirs(IMG_DIR, exist_ok=True)

images = []
for item in parse_json_gz(META_PATH):
    if len(images) >= N:
        break
    asin = item.get('asin', '')
    img_url = item.get('imUrl', '')
    if not asin or not img_url:
        continue
    path = download_image(asin, img_url, IMG_DIR)
    if path is None:
        continue
    img = load_image(path)
    if img is not None:
        images.append(img)

print(f'Collected {len(images)} images')

Collected 10 images


In [4]:
(resnet, resnet_tf), (clip_model, clip_proc), (dino_model, dino_proc) = build_models(device)

[models] Loading ResNet50 ...
[models] Loading CLIP ...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

[models] Loading DINOv2 ...


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[models] All models loaded.



In [ ]:
import torch
from image_embeddings import embed_resnet, embed_clip, embed_dinov2

def measure(fn, *args, n_runs=3):
    with torch.no_grad():
        fn(*args)
    if device.type == 'cuda':
        torch.cuda.synchronize()

    times = []
    for _ in range(n_runs):
        if device.type == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            fn(*args)
        if device.type == 'cuda': torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

    avg = sum(times) / len(times)
    return avg

In [9]:
t_resnet = measure(embed_resnet, images, resnet, resnet_tf, device)
t_clip   = measure(embed_clip,   images, clip_model, clip_proc, device)
t_dino   = measure(embed_dinov2, images, dino_model, dino_proc, device)

print(f'\nResults over {N} images (average of 3 runs):\n')
print(f'ResNet50 : {t_resnet*1000:.1f} ms  ({t_resnet/N*1000:.1f} ms/image)')
print(f'CLIP     : {t_clip*1000:.1f} ms  ({t_clip/N*1000:.1f} ms/image)')
print(f'DINOv2   : {t_dino*1000:.1f} ms  ({t_dino/N*1000:.1f} ms/image)')


Results over 10 images (average of 3 runs):

ResNet50 : 21.4 ms  (2.1 ms/image)
CLIP     : 32.7 ms  (3.3 ms/image)
DINOv2   : 61.8 ms  (6.2 ms/image)
